In [1]:
from Model import *
from Dataset import *
from Perfomance import *

I0000 00:00:1776413467.786783   55672 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


There is all testing for modules provided

In [2]:
#Testing functions of Model class
model = CNNLSTM()

model.compile(optimizer='SGD', loss={
        'binary': binary_loss_fn,
        'multiclass': multiclass_loss_fn
    })

model([np.random.rand(5,8,8,112), np.random.rand(5)])
#model.summary()
ar = np.load(f'{DATA_DIR}/test.npz')
positions = ar['x']; evals = ar['evals'].astype(np.float32); target = ar['y']
#print(positions.shape, evals.shape, target.shape)
with tf.GradientTape() as tape:
    preds = model.binary_call((positions, evals))
    loss = binary_loss_fn(target, preds)
print(loss)

E0000 00:00:1776413472.001680   55672 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/demid/Code/Python/ChessErrorClassification/venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'SGD', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


tf.Tensor(0.017385453, shape=(), dtype=float32)


In [3]:
#Testing Dataset
ds = build_binary_dataset(100)
model.training_run(ds, batch_size=5)

I0000 00:00:1776413478.140114   55672 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


tf.Tensor(0.010695137, shape=(), dtype=float32)
Batch 0 | Loss 0.010695137083530426 | Balanced Accuracy 0
tf.Tensor(0.011545926, shape=(), dtype=float32)
tf.Tensor(0.008491134, shape=(), dtype=float32)
tf.Tensor(0.008005791, shape=(), dtype=float32)
tf.Tensor(0.010019814, shape=(), dtype=float32)
tf.Tensor(0.007894378, shape=(), dtype=float32)
Batch 5 | Loss 0.007894378155469894 | Balanced Accuracy 0


KeyboardInterrupt: 

In [4]:
#Testing Perfomance
batch_size=10

positions = np.random.randn(batch_size,4,8,8,112)
evals = np.random.randn(batch_size,4)
target = np.zeros(shape=(batch_size,num_classes))
for i in range(batch_size):
    if np.random.rand()<=class_weight:
        class_value = num_classes-1
    else:
        class_value = np.random.randint(low=0, high=num_classes-1)
    target[i, class_value]=1

'''ar = np.load("data/batch0.npz")
positions = ar['x'][:batch_size]
target = ar['y'][:batch_size]
    
evals = ar['evals'][:batch_size]
print(positions.shape, target.shape, evals.shape)
target = np.append(target, np.zeros((target.shape[0], 1)), axis=1) #For dimensionality match'''
bin_target = tf.cast(tf.equal(target[:, -1], 0), tf.int8)
model.only_bin = False
with tf.GradientTape() as tape:
    preds = model((positions, evals))
    binary_loss = binary_loss_fn(bin_target, preds['binary'])
    multiclass_loss = multiclass_loss_fn(target, preds['multiclass'])
    loss = binary_loss+multiclass_loss

grad = tape.gradient(loss, model.trainable_variables)
#print(f"\n\n Preds are {preds} \n\n")

binary_auc = BinaryAUCMetric()
binary_auc.update_state(target, preds)
print(f"Binary AUC is {binary_auc.result()}")
mean_loss = LossMetric()
mean_loss.update_state(loss)
print(f"Mean loss is unexpectedly {mean_loss.result()}")
#model.evaluate()
binary_acc = BinaryAccuracyMetric()
binary_acc.update_state(target, preds)
print(f"Binary accuracy is {binary_acc.result()}")
multiclass_acc = AccuracyMetric()
multiclass_acc.update_state(target, preds)
print(f"Class prediction accuracy is {multiclass_acc.result()}")

#model.evaluate(x=(positions, evals), y={'binary':target, 'multiclass':target},verbose=1)

Binary AUC is 0.6161664724349976
Mean loss is unexpectedly 1.7859852313995361
Binary accuracy is 0.5
Class prediction accuracy is 0.0
